# Top-10 prediction agreement: `original_logi` vs `new_logits`

Both come from the same contribution decomposition. `original_logi` is the in-trace bf16 logits; `new_logits` is the float32 `calc_logits` path applied to the summed final-layer contributions. This checks the top-10 next-token list (order included) matches across several fresh prompts.

In [1]:
from pathlib import Path

from api_checks.api_cache import APICache
from info_flow.config import Config
import torch

config = Config()
api_cache = APICache(hf_token=config.hf_token, cache_path=Path(config.result_cache_path))
calculator = api_cache.get_infomration_calculator(config.info_flow_model)

C:\Users\wildn\Desktop\New life\AI\Expirments\VisulaiztionInfoFlowDemo\real\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROMPTS = [
    "The capital of France is",
    "Water is composed of hydrogen and",
    "The opposite of hot is",
    "In 1969, the first humans landed on the",
    "The mitochondria is the powerhouse of the",
    "Roses are red, violets are",
    "The square root of sixteen is",
    "She opened the door and saw a",
    "Python is a popular programming",
    "The sun rises in the",
]
TOP_K = 10

In [3]:
all_match = True
for i, prompt in enumerate(PROMPTS):
    full = api_cache.get_full_run_results(config.info_flow_model, prompt)
    original_logi = full.logits  # (p_len, vocab) in-trace bf16 logits
    last_mlp_output = full.contributions.post_mlp_contribution[-1].sum(dim=1)  # (p_len, d_model)
    new_logits = calculator.calc_logits(last_mlp_output)  # (p_len, vocab) float32 path

    orig_top = list(calculator.calc_top_probabilities_from_logits(original_logi[-1], TOP_K).keys())
    new_top = list(calculator.calc_top_probabilities_from_logits(new_logits[-1], TOP_K).keys())

    ordered_match = orig_top == new_top
    set_match = set(orig_top) == set(new_top)
    all_match &= ordered_match

    status = "OK " if ordered_match else ("SET" if set_match else "DIFF")
    print(f"\n[{i:2d}] {status}  {prompt!r}")
    if not ordered_match:
        print(f"     orig: {orig_top}")
        print(f"     new : {new_top}")
    else:
        print(f"     top10: {orig_top}")

print("\n" + ("ALL PROMPTS: top-10 lists match (order included)." if all_match else "MISMATCH found - see above."))

⬇ Downloading:   0%|          | 0.00/57.1M [00:00<?]

⬇ Downloading:   0%|          | 131k/57.1M [00:00<02:02]

⬇ Downloading:   0%|          | 262k/57.1M [00:00<01:27]

⬇ Downloading:   1%|          | 393k/57.1M [00:00<01:14]

⬇ Downloading:   1%|          | 655k/57.1M [00:00<00:49]

...

⬇ Downloading:  97%|█████████▋| 55.6M/57.0M [00:05<00:00]

⬇ Downloading: 100%|█████████▉| 56.9M/57.0M [00:05<00:00]

⬇ Downloading: 100%|██████████| 57.0M/57.0M [00:05<00:00]

Result saved in cache sucessfully
torch.Size([6, 4096])
torch.Size([6, 1])



[ 9] OK   'The sun rises in the'
     top10: [' east', ' East', ' sky', ' morning', ' west', ' eastern', ' south', ' north', ' distance', ' West']

MISMATCH found - see above.
